# Deep Agents and Voice

*A realtime voice assistant that hands the hard questions to a deep agent.*

You talk; it talks back. For small talk it just answers. But when you ask something that needs real
research, it delegates to a **deep agent** &mdash; planning, web search, report writing &mdash; and then
narrates the findings back to you, conversationally.

This is the notebook-native retelling of
[**langchain-ai/google-adk-realtime-deepagents-example**](https://github.com/langchain-ai/google-adk-realtime-deepagents-example),
which wires the same idea into a browser app with Google ADK, FastAPI, and WebSocket audio. Here we drop
the web stack and run every piece in the kernel, so you can watch each part do its job.

<pre style="border:1px solid #999; border-radius:6px; padding:12px 14px; line-height:1.4;">
  &#127908;  you speak  &#8594;  <b>Gemini Live</b>  &#8594;  &#128266; you hear the reply
                              &#9474;
             small talk?  &#8594; answers directly
                              &#9474;
             research-y?  &#8594; calls <b style="color:#2b8a3e;">deep_research(topic)</b>
                              &#9474;
                              &#9660;
                    <b>Deep agent</b> (Claude + Tavily)
                     &#8226; plan  &#8226; search the web &#215;N  &#8226; write a spoken report
                              &#9474;
                              &#9660;
             report  &#8594;  back to Gemini Live  &#8594;  narrated aloud
</pre>

The voice layer is **Gemini Live** &mdash; bidirectional audio with server-side voice-activity detection,
so you just talk and it knows when you have stopped. The research brain is a **deepagents** agent backed
by **Claude** and **Tavily**, exposed to the voice model as a single `deep_research` tool.

> **Run this locally.** Live mic capture needs a real microphone and speakers, so use a local kernel
> (not a remote/Colab one). macOS will ask for microphone permission the first time you start the loop.

In [1]:
%load_ext autoreload
%autoreload 2

# Setup

Load the environment (`GEMINI_API_KEY` for the voice layer, `ANTHROPIC_API_KEY` and `TAVILY_API_KEY`
for the research brain), import what we need, and pick the two models.

In [2]:
import os

from dotenv import load_dotenv

load_dotenv(override=True)

import asyncio

from google import genai
from google.genai import types
from langchain_core.tools import tool
from tavily import TavilyClient
from deepagents import create_deep_agent
import ipywidgets as widgets
from IPython.display import display

from util import MicInput, SpeakerOutput, MIC_RATE, reset_audio, LiveActivityPanel, print_exchange

model = "claude-sonnet-5"                      # the research brain (Anthropic)
LIVE_MODEL = "gemini-3.1-flash-live-preview"   # the voice layer (Gemini Live)

# Point the voice client straight at Google. If a LangChain LLM gateway is configured
# (GOOGLE_GEMINI_BASE_URL), it proxies REST fine but NOT the Live WebSocket — so a
# default client would fail the handshake with HTTP 403. Passing base_url explicitly
# bypasses the gateway for Gemini Live; Claude and Tavily below are unaffected.
# We also pass the key explicitly so an ambient GOOGLE_API_KEY can't shadow GEMINI_API_KEY.
client = genai.Client(
    api_key=os.environ.get("GEMINI_API_KEY") or os.environ["GOOGLE_API_KEY"],
    http_options=types.HttpOptions(base_url="https://generativelanguage.googleapis.com/"),
)

# Model ids for Live move fast. If connecting errors with "model not found", list the
# current ones — `[m.name for m in client.models.list() if "live" in m.name]` — and swap
# LIVE_MODEL. "gemini-2.5-flash-native-audio-preview-09-2025" is one alternative.

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


# The research brain

Before any audio, let's build the part that does the actual work. A deep agent comes with planning
(`write_todos`), a virtual filesystem, and subagents out of the box &mdash; and here we lean into two of
those on purpose. The **coordinator** plans the work as a todo list, then **delegates** the searching to
a `researcher` subagent that owns the Tavily tool. That structure isn't just tidy: it gives us concrete,
streamable activity &mdash; a plan, a hand-off, each search &mdash; to put on screen while the caller
waits. Both prompts are tuned for **spoken** answers: short, plain sentences, no markdown or URLs to read
aloud.

In [3]:
@tool
def internet_search(query: str, max_results: int = 5) -> list[dict]:
    """Search the web for information.

    Args:
        query: what to search for
        max_results: how many results to return
    """
    response = TavilyClient().search(query, max_results=max_results)
    return response.get("results", [])


# A subagent that owns the searching. Giving the Tavily tool ONLY to the subagent means
# the coordinator can't search on its own — it has to plan and then delegate. That's the
# machinery we want on screen: a todo plan, a hand-off, and the searches streaming in
# while the caller waits for an answer. The search cap keeps a live demo snappy (and the
# activity panel readable) — a broad question can otherwise trigger a dozen-plus searches.
researcher = {
    "name": "researcher",
    "description": "Searches the web on a focused question and returns concise findings.",
    "system_prompt": (
        "You are a focused web researcher. Run at most 4-5 targeted searches — no more — "
        "cross-check claims across sources, then stop and return concise findings in plain "
        "sentences. No markdown or URLs."
    ),
    "tools": [internet_search],
}

RESEARCH_INSTRUCTIONS = """You are a research coordinator answering questions that will be READ ALOUD.

First call write_todos to lay out a short plan (2-4 steps). Then delegate the searching to the
`researcher` subagent with the task tool, giving it complete, self-contained instructions in one call.
Update the todos as the work progresses. When the findings come back, write a short spoken report:
3-5 sentences of plain, conversational language. No markdown, no bullet lists, no URLs. Lead with the
answer; name a source only when it genuinely matters."""

research_agent = create_deep_agent(
    model=model,
    system_prompt=RESEARCH_INSTRUCTIONS,
    subagents=[researcher],
)

We'll call the agent the same way from inside the voice loop, so wrap it in a small coroutine that
returns just the final report as plain text (Anthropic replies can arrive as a list of content blocks,
so flatten those to a string). Instead of a single `ainvoke`, we **stream** the run with `subgraphs=True`
and push every step to a live `LiveActivityPanel` &mdash; the coordinator's todo plan, the hand-off to
the `researcher` subagent, and each web search &mdash; so there's something to watch while the (blocking)
research runs. Coordinator events arrive with an empty namespace `ns`; the subagent's searches arrive
under a nested one, which is how we attribute each step to the right actor.

In [4]:
def _text(content) -> str:
    """Flatten LangChain message content (str, or a list of Anthropic blocks) to text."""
    if isinstance(content, list):
        return "".join(b.get("text", "") for b in content if isinstance(b, dict))
    return content or ""


async def run_research(topic: str, panel=None) -> str:
    """Run the deep agent and return its final report, streaming each step to `panel`.

    We stream (updates + subgraphs) instead of a single ainvoke so the caller can watch
    the internals while the blocking research runs: the coordinator's plan (write_todos),
    the hand-off to the researcher subagent (task), and each web search. Top-level events
    arrive with an empty namespace `ns` (the coordinator); a non-empty `ns` is the
    subagent's own subgraph — that's how we know who is doing what.
    """
    report = ""
    current_sub = None
    async for ns, update in research_agent.astream(
        {"messages": [{"role": "user", "content": topic}]},
        stream_mode="updates",
        subgraphs=True,
    ):
        scope = "coordinator" if not ns else (current_sub or "researcher")
        for payload in (update or {}).values():
            messages = payload.get("messages", []) if isinstance(payload, dict) else []
            for m in messages:
                for call in getattr(m, "tool_calls", None) or []:
                    name, args = call["name"], (call["args"] or {})
                    if panel is None:
                        continue
                    if name == "write_todos":
                        panel.plan(scope, args.get("todos", []))
                    elif name == "task":
                        current_sub = args.get("subagent_type", "subagent")
                        panel.delegate(current_sub)
                    elif name == "internet_search":
                        panel.search(scope, args.get("query", ""))
                # the coordinator's final spoken report (top-level namespace)
                if not ns and m.__class__.__name__ == "AIMessage":
                    text = _text(m.content)
                    if text.strip():
                        report = text
    return report or "I couldn't find anything useful on that."

# Wrapping research as a Live tool

The Live API doesn't auto-run your Python functions the way the regular API can &mdash; you **declare**
the tool as a schema, and when the model decides to call it you run it yourself and send the result back.
So we describe `deep_research` to Gemini Live as a function that takes a `topic`.

**Blocking vs. non-blocking.** By default a tool call is *blocking*: the model goes quiet until we return
the report, then narrates it. The source repo makes the call `NON_BLOCKING` so the assistant can keep
chatting while research runs in the background (add `behavior=types.Behavior.NON_BLOCKING` to the
declaration). We keep the default blocking behavior here &mdash; it's simpler and reliable across Live
models.

In [6]:
deep_research_tool = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name="deep_research",
            description=(
                "Run in-depth web research on a topic and return a short, spoken-friendly report. "
                "Use whenever a good answer needs current facts, figures, comparisons, or multiple sources."
            ),
            parameters=types.Schema(
                type=types.Type.OBJECT,
                properties={
                    "topic": types.Schema(
                        type=types.Type.STRING,
                        description="the question or topic to research",
                    )
                },
                required=["topic"],
            ),
        )
    ]
)

# The voice layer &mdash; Gemini Live

Now the voice session config. Gemini Live streams audio both ways: we send **16 kHz** PCM from the mic,
it sends **24 kHz** PCM back. Server-side voice-activity detection decides when you've finished speaking,
and turning on input/output transcription gives us live captions to print. The system prompt tells the
model to stay brief, answer small talk directly, and reach for `deep_research` when a question needs it.

In [7]:
VOICE_INSTRUCTIONS = """You are a warm, concise voice assistant. Keep replies short and natural —
you are being heard, not read.

You are NOT a general-knowledge oracle. For ANY question involving facts, current events, numbers,
comparisons, or anything described as latest/recent — or anything you can't answer with complete
confidence from memory — you MUST call the deep_research tool with a clear topic instead of answering
from your own knowledge. Only answer directly for pure small talk (greetings, chit-chat, clarifying
questions).

When you call deep_research, give a brief spoken acknowledgement first (like "Sure, let me look into
that"), and when the report comes back, weave it into a natural spoken answer and offer to go deeper."""

live_config = types.LiveConnectConfig(
    system_instruction=VOICE_INSTRUCTIONS,
    response_modalities=["AUDIO"],
    tools=[deep_research_tool],
    input_audio_transcription=types.AudioTranscriptionConfig(),
    output_audio_transcription=types.AudioTranscriptionConfig(),
    speech_config=types.SpeechConfig(
        voice_config=types.VoiceConfig(
            prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name="Puck")
        )
    ),
)

# Audio plumbing

Capturing the mic and playing audio back is boilerplate, so it lives in `util/voice.py`:

- **`MicInput`** opens a 16 kHz input stream and hands each PCM frame to asyncio.
- **`SpeakerOutput`** plays the model's 24 kHz PCM gaplessly, and its `flush()` drops queued audio for
  **barge-in** &mdash; when you interrupt the assistant, its half-spoken sentence stops instead of
  talking over you.

Both are context managers, so the device streams always close cleanly when the loop ends.

# The realtime loop

This is the whole thing. Inside one Live session we run two coroutines at once:

- **uplink** &mdash; forward mic frames to the model (paused while the assistant is speaking &mdash; see below).
- **downlink** &mdash; for each message: play audio, print transcripts, handle barge-in, and when the
  model calls `deep_research`, run the deep agent &mdash; **streaming each step into a live activity
  panel** so you can watch it plan, delegate, and search &mdash; then send the report back to be narrated.

Below the transcript sits a **`LiveActivityPanel`**: the moment research starts it shows the
coordinator's todo plan, the hand-off to the `researcher` subagent, and every web search as it fires,
with a ticking `⏳ Ns` timer so there's always motion &mdash; no more silent dead air while you wait.
A **Stop** button (and a safety time cap) end the session cleanly.

> **Use headphones.** On laptop speakers, the mic picks up the assistant's own voice and Gemini's
> voice-activity detection reads that as you interrupting &mdash; which cuts the reply off after a word
> or two. To avoid that without headphones we run **half-duplex**: the mic is muted while the assistant
> is speaking (the `loop.time() - last_audio_at` check below). With headphones you can drop that gate for
> true barge-in.

**How to use it:** run the cell, allow microphone access when macOS asks, then talk. Small talk
(*"how's it going?"*) is answered directly. A research question
(*"what's the latest on solid-state EV batteries?"*) lights up the activity panel &mdash; plan &rarr;
delegate &rarr; searches, ticking away &mdash; then you hear the spoken report. Click **Stop** when
you're done.

In [ ]:
STOP = asyncio.Event()
MAX_SECONDS = 180  # safety cap so the session always ends

stop_button = widgets.Button(description="Stop", button_style="danger", icon="stop")
stop_button.on_click(lambda _: STOP.set())
transcript = widgets.HTML()
display(stop_button, transcript)

# Live "what the deep agent is doing right now" panel. It renders in place below the
# transcript: the coordinator's todo plan, the delegation to the researcher subagent,
# each web search, and a ticking elapsed timer so there's motion even between steps.
activity = LiveActivityPanel()


def log(role, text=""):
    # Big type so the transcript reads from the back of the room (matches the panel).
    transcript.value += f"<div style='margin:8px 0;font-size:24px;line-height:1.5;'><b>{role}</b> {text}</div>"


async def voice_session():
    STOP.clear()
    transcript.value = ""
    user_buf, asst_buf = [], []
    loop = asyncio.get_running_loop()
    last_audio_at = 0.0  # when the assistant last sent audio — drives the half-duplex mic gate

    # Release any mic/speaker device left wedged by a previous interrupted run, so a live
    # demo can just re-run this cell after a Stop/interrupt instead of restarting the kernel.
    reset_audio()

    async with client.aio.live.connect(model=LIVE_MODEL, config=live_config) as session:
        with MicInput() as mic, SpeakerOutput() as speaker:

            async def uplink():
                async for frame in mic.frames():
                    # Half-duplex: stay quiet for a beat after the assistant's audio so the mic
                    # doesn't echo the speakers back and trigger a false "barge-in" that cuts the
                    # reply off. Use headphones and you can drop this gate for true barge-in.
                    if loop.time() - last_audio_at < 0.5:
                        continue
                    await session.send_realtime_input(
                        audio=types.Blob(data=frame, mime_type=f"audio/pcm;rate={MIC_RATE}")
                    )

            async def downlink():
                nonlocal last_audio_at
                async for msg in session.receive():
                    if msg.data:
                        speaker.play(msg.data)
                        last_audio_at = loop.time()

                    sc = msg.server_content
                    if sc:
                        if sc.interrupted:
                            speaker.flush()  # real barge-in (e.g. on headphones): drop stale speech
                        if sc.input_transcription and sc.input_transcription.text:
                            user_buf.append(sc.input_transcription.text)
                        if sc.output_transcription and sc.output_transcription.text:
                            if user_buf:
                                log("🧑 You:", "".join(user_buf)); user_buf.clear()
                            asst_buf.append(sc.output_transcription.text)
                        if sc.turn_complete and asst_buf:
                            log("🤖 Assistant:", "".join(asst_buf)); asst_buf.clear()

                    if msg.tool_call:
                        for fc in msg.tool_call.function_calls:
                            if user_buf:
                                log("🧑 You:", "".join(user_buf)); user_buf.clear()
                            topic = (fc.args or {}).get("topic", "")
                            log("🔎 Researching:", topic)
                            # Drive the live panel: plan → delegate → searches, with a ticking timer,
                            # so the audience sees the deep agent working while we wait for the report.
                            activity.start(topic)
                            report = await run_research(topic, activity)
                            activity.finish(report)
                            await session.send_tool_response(
                                function_responses=[
                                    types.FunctionResponse(id=fc.id, name=fc.name, response={"report": report})
                                ]
                            )

            tasks = [asyncio.create_task(uplink()), asyncio.create_task(downlink())]
            try:
                await asyncio.wait_for(STOP.wait(), timeout=MAX_SECONDS)
            except asyncio.TimeoutError:
                pass
            finally:
                for t in tasks:
                    t.cancel()
                await asyncio.gather(*tasks, return_exceptions=True)
                activity.finish()  # stop the heartbeat if a research call was cut off mid-run


await voice_session()

Button(button_style='danger', description='Stop', icon='stop', style=ButtonStyle())

HTML(value='')

HTML(value='')

# Recap

- **Same idea, no web stack.** A realtime voice model that delegates hard questions to a deep agent —
  driven straight from `google-genai`'s Live API instead of Google ADK + FastAPI + a browser frontend.
- **The deep agent is just a tool.** We declared `deep_research` to Gemini Live, and on a tool call ran a
  `create_deep_agent` (Claude + Tavily) and streamed the report back for the model to narrate.
- **Blocking vs. non-blocking.** We used a blocking tool call for reliability; the source repo goes
  non-blocking so the voice keeps flowing while research runs.
- **Server VAD + barge-in.** We stream mic audio continuously and let the model detect turn ends;
  `SpeakerOutput.flush()` handles interruptions.
- **What we skipped vs. the repo:** the FastAPI/WebSocket transport, the browser AudioWorklets, and
  nesting the whole conversation under one LangSmith `RunTree`. Each `deep_research` call still shows up
  as its own deep-agent trace in LangSmith when `LANGSMITH_TRACING=true`.